In [ ]:
import os
import yaml

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
echonext_data = "/opt/gpudata/ecg/echonext"
subset_root = "/opt/gpudata/ecg/temp" # path to make subset directories

In [ ]:
wv_full = np.load(os.path.join(echonext_data, "EchoNext_train_waveforms.npy"))
tb_full = np.load(os.path.join(echonext_data, "EchoNext_train_tabular_features.npy"))

In [ ]:
_df = pd.read_csv(os.path.join(echonext_data, "EchoNext_metadata_100k.csv"))
all_train_samples = _df["split"] == "train"
val_test_nosplit = _df[~all_train_samples]
df_full = _df[all_train_samples]

with open("../configs/targets.yaml", "r") as f:
    cfg = yaml.safe_load(f)

# too many rare classes if we stratify by all labels, so just use the composite
# target_cols = list(cfg["target_columns"].values())
target_cols = [cfg["target_columns"][cfg["composite_target"]]]

In [ ]:
def make_splitter(df):
    splitting_df = df[["age_at_ecg", "sex"] + target_cols].copy()
    splitting_df["sex"] = pd.Categorical(splitting_df["sex"]).codes
    splitting_df["age_at_ecg"] = pd.qcut(splitting_df["age_at_ecg"], q=4).cat.codes
    splitter = splitting_df.apply(lambda row: "_".join(row.astype(str)), axis="columns")
    return splitter

In [ ]:
DF_WV_TB_T = tuple[pd.DataFrame, np.ndarray, np.ndarray]

def make_train_subset(src_data: DF_WV_TB_T, tgt_size: int) -> DF_WV_TB_T:
    # make subsets of training data
    df_src, wv_src, tb_src = src_data
    tgt_size_k = tgt_size // 1024
    if tgt_size_k == 0:
        tgt_size_k = str(tgt_size)
    else:
        tgt_size_k = f"{tgt_size_k}k"
    subset_path = os.path.join(subset_root, f"echonext-{tgt_size_k}")
    os.makedirs(subset_path, exist_ok=True)

    df_tgt, _, wv_tgt, _, tb_tgt, _ = train_test_split(
        df_src,
        wv_src,
        tb_src,
        train_size=tgt_size,
        random_state=42,
        shuffle=True,
        stratify=make_splitter(df_src),
    )

    df = pd.concat([df_tgt, val_test_nosplit])
    # keep the name "100k" consistent as the filenames within the echonext folder are hardcoded elsewhere in our codebase
    df.to_csv(os.path.join(subset_path, "EchoNext_metadata_100k.csv"), index=False)

    np.save(os.path.join(subset_path, "EchoNext_train_waveforms.npy"), wv_tgt)
    np.save(os.path.join(subset_path, "EchoNext_train_tabular_features.npy"), tb_tgt)

    # also link val/test split data
    for p in [
        "EchoNext_val_waveforms.npy",
        "EchoNext_val_tabular_features.npy",
        "EchoNext_test_waveforms.npy",
        "EchoNext_test_tabular_features.npy",
    ]:
        os.symlink(
            src=os.path.join(echonext_data, p),
            dst=os.path.join(subset_path, p),
        )

    return df_tgt, wv_tgt, tb_tgt

In [ ]:
curr = (df_full, wv_full, tb_full)
for tgt_size in tqdm([32768, 16384, 8192, 4096, 2048, 1024, 512, 256]):
    curr = make_train_subset(curr, tgt_size)